# Detecção de fraude em transações de cartão de crédito com Spark ML

**Disciplina:** Processamento de Dados Massivos — Prof. Alexandre Roriz  
**Tipo de problema sorteado:** Classificação (binária, com forte desbalanceamento)  
**Autor:** _<seu nome>_  

**Pipeline:** ingestão → limpeza → engenharia de atributos → modelagem → avaliação → persistência

**Requisitos:** Google Colab (ou Python 3.9+ com Java e `pip install pyspark pandas matplotlib`). A base tem ~150 MB e é baixada pelo próprio notebook (nenhum dado vai para o GitHub; adicione `dados/` e `modelos/` ao `.gitignore`).

## 1. Contextualização

### 1.1 A base
- **Fonte original:** *Credit Card Fraud Detection*, ULB Machine Learning Group (Worldline e Université Libre de Bruxelles), publicada no Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud (também disponível no OpenML, id 1597). Referência: Dal Pozzolo et al., *Calibrating Probability with Undersampling for Unbalanced Classification*, IEEE SSCI, 2015.
- **Link de leitura (reprodutível, sem login):** o Kaggle exige conta, então o notebook usa a cópia pública hospedada pelo TensorFlow: `https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv`
- **Período:** transações de titulares de cartões europeus ao longo de **dois dias de setembro de 2013**.
- **Número de registros e colunas:** calculados na seção 2 (284.807 transações e 31 colunas: `Time`, `V1`–`V28`, `Amount` e `Class`).
- **Amostra de um conjunto maior?** Sim. A base é um recorte de dois dias de transações reais de uma instituição. O volume total do portfólio de origem não foi divulgado. As variáveis `V1`–`V28` são componentes de uma **PCA** aplicada pelos autores por confidencialidade, e apenas `Time` e `Amount` estão no formato original. A classe positiva (fraude) tem 492 casos, **0,172%** do total.

### 1.2 Pergunta de negócio
> Dada uma transação com cartão, **ela é uma fraude?**

Bancos e operadoras precisam bloquear ou revisar transações suspeitas **em tempo real**. Cada fraude não detectada gera prejuízo direto, e cada alerta falso gera custo de revisão e incômodo ao cliente. O rótulo é `Class` (1 = fraude, 0 = legítima).

### 1.3 Por que é um problema de dados massivos?
A base de exemplo é pequena, mas o **cenário** que ela representa não é, e é isso que justifica o processamento distribuído:
- **Volume:** redes de cartões e grandes bancos processam **bilhões de transações por ano**. Como a fraude é rara (aqui, 0,17%), para reunir dezenas de milhares de exemplos positivos para treino é preciso processar centenas de milhões de transações. A raridade da classe de interesse **multiplica o volume** necessário.
- **Velocidade:** a decisão precisa sair em milissegundos, sobre um fluxo contínuo de transações, e o modelo precisa ser reavaliado e retreinado com frequência, porque os fraudadores mudam de comportamento (*concept drift*). É um caso típico de *streaming* e de processamento incremental.
- **Variedade e atributos derivados:** em produção, atributos importantes (número de transações do cartão na última hora, valor médio dos últimos 30 dias, país da transação) vêm de agregações por janela sobre todo o histórico, o que exige *shuffles* e *window functions* distribuídas.
- **Custo computacional do treino:** lidar com desbalanceamento (pesos, validação cruzada estratificada, ajuste de limiar) multiplica o número de treinos. Em escala real, isso só é viável com processamento paralelo.

> Aqui o Spark roda em modo local sobre uma amostra de 284.807 transações. O código é o mesmo que seria executado em um cluster sobre o histórico completo, bastando trocar a leitura e o `master`.

## 2. Ingestão
Os dados são baixados da fonte pública. O **schema é definido manualmente** com `StructType` (todas as colunas são numéricas), o que evita uma passagem extra pelo arquivo para inferir tipos e garante os tipos corretos. O cabeçalho do arquivo é conferido contra o schema.

In [1]:
# Instalação do PySpark (necessária no Google Colab; o Java já vem instalado lá)
import subprocess, sys
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
print("PySpark", pyspark.__version__)

PySpark 4.0.5


In [2]:
import os, json, datetime, urllib.request
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array
import matplotlib.pyplot as plt

spark = (SparkSession.builder
         .appName("fraude-cartao")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.0.5


In [3]:
URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
PASTA = os.path.abspath("dados")
CAMINHO = f"{PASTA}/creditcard.csv"
os.makedirs(PASTA, exist_ok=True)

if not os.path.exists(CAMINHO):
    print("Baixando (~150 MB)...")
    urllib.request.urlretrieve(URL, CAMINHO + ".part")
    os.replace(CAMINHO + ".part", CAMINHO)
    print("Download concluído em", datetime.date.today())
else:
    print("Arquivo já existe:", CAMINHO)
print(f"Tamanho do arquivo: {os.path.getsize(CAMINHO) / 1e6:.0f} MB")

Baixando (~150 MB)...
Download concluído em 2026-11-03
Tamanho do arquivo: 151 MB


In [4]:
# Schema definido manualmente
schema = StructType(
    [StructField("Time", DoubleType())] +
    [StructField(f"V{i}", DoubleType()) for i in range(1, 29)] +
    [StructField("Amount", DoubleType()), StructField("Class", DoubleType())]
)

# Confere o cabeçalho do arquivo com o schema
cabecalho = spark.read.csv(CAMINHO, header=True).columns
esperado = [f.name for f in schema.fields]
assert cabecalho == esperado, f"Cabeçalho difere do schema:\n{cabecalho}\n{esperado}"

raw = spark.read.csv(CAMINHO, header=True, schema=schema, mode="PERMISSIVE")
raw.printSchema()
n_raw = raw.count()
print(f"Registros: {n_raw:,} | Colunas: {len(raw.columns)}")

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

## 3. Exploração e limpeza
Toda a exploração usa funções do Spark.

In [5]:
# Estatísticas descritivas
raw.select("Time", "V1", "V2", "V3", "Amount", "Class").describe().show()

# Nulos por coluna (soma de nulos em todas as colunas)
raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).show(vertical=True)

# Distribuição do alvo
raw.groupBy("Class").count().withColumn("proporcao", F.round(F.col("count") / n_raw, 5)).show()

# Valor das transações por classe
raw.groupBy("Class").agg(F.round(F.avg("Amount"), 2).alias("media"),
                         F.expr("percentile_approx(Amount, 0.5)").alias("mediana"),
                         F.max("Amount").alias("maximo")).show()

+-------+------------------+--------------------+--------------------+--------------------+------------------+--------------------+
|summary|              Time|                  V1|                  V2|                  V3|            Amount|               Class|
+-------+------------------+--------------------+--------------------+--------------------+------------------+--------------------+
|  count|            284807|              284807|              284807|              284807|            284807|              284807|
|   mean| 94813.85957508067|5.109395214431826...|2.554697607215913...|-8.63487791238978...|  88.3496192509508|0.001727485630620034|
| stddev|47488.145954566906|  1.9586958038574882|   1.651308579476995|  1.5162550051777774|250.12010924018736| 0.04152718963546528|
|    min|               0.0|    -56.407509631329|   -72.7157275629303|   -48.3255893623954|               0.0|                 0.0|
|    max|          172792.0|    2.45492999121121|    22.0577289904909|    9.

In [6]:
# Duplicatas e valores inválidos (antes de tratar)
n_dist = raw.dropDuplicates().count()
print(f"Linhas duplicadas: {n_raw - n_dist:,}")

raw.select(
    F.sum((F.col("Amount") < 0).cast("int")).alias("valor_negativo"),
    F.sum((F.col("Time") < 0).cast("int")).alias("tempo_negativo"),
    F.sum((~F.col("Class").isin(0.0, 1.0) | F.col("Class").isNull()).cast("int")).alias("classe_invalida"),
    F.sum((F.col("Amount") == 0).cast("int")).alias("valor_zero"),
).show(vertical=True)

Linhas duplicadas: 1,081
-RECORD 0---------------
 valor_negativo  | 0    
 tempo_negativo  | 0    
 classe_invalida | 0    
 valor_zero      | 1825 



### Regras de limpeza
1. **Duplicatas:** removidas com `dropDuplicates()`. Repetir a mesma linha inflaria o treino e poderia colocar a mesma transação no treino **e** no teste (vazamento).
2. **Valores inválidos:** `Amount` ou `Time` negativos e `Class` fora de {0, 1} seriam descartados (a verificação acima mostra se existem).
3. **Nulos:** descartados, se houver (a base original não costuma ter).
4. Transações com `Amount = 0` são **mantidas**: são válidas (verificação de cartão) e podem ter relação com fraude.

In [7]:
contagens = {"1. Original": n_raw}

df = raw.dropDuplicates()
contagens["2. Sem duplicatas"] = df.count()

df = (df.dropna()
        .filter((F.col("Amount") >= 0) & (F.col("Time") >= 0) & F.col("Class").isin(0.0, 1.0)))
contagens["3. Sem nulos/valores inválidos"] = df.count()

for k, v in contagens.items():
    print(f"{k:34s} {v:>10,}")

1. Original                           284,807
2. Sem duplicatas                     283,726
3. Sem nulos/valores inválidos        283,726


## 4. Engenharia de atributos
- **Rótulo:** `label = Class`.
- **Temporais:** `Time` é o tempo em segundos desde a primeira transação (não generaliza para outros períodos). Dele derivamos `hora` do dia, codificada de forma cíclica (`hora_sin`, `hora_cos`) e a categoria `periodo_dia` (madrugada, manhã, tarde, noite).
- **Valor:** `log_valor = log(1 + Amount)`, pois `Amount` é muito assimétrico (muitas compras pequenas e poucas muito grandes), e a categoria `faixa_valor` (zero, até 10, 10–100, 100–500, acima de 500). Os limites são **constantes fixas**, não calculadas dos dados, o que evita vazamento.
- **Originais:** `V1`–`V28` (componentes PCA) usados como estão.
- **Categóricas** (`StringIndexer` + `OneHotEncoder`): `periodo_dia`, `faixa_valor`.
- **Numéricas** (`VectorAssembler` + `StandardScaler`): `V1`–`V28`, `log_valor`, `hora_sin`, `hora_cos`. Tudo em um **Pipeline**, ajustado **somente no treino**.
- **Excluídos:** `Time` bruto (relativo ao início da coleta) e `Amount` bruto (substituído por `log_valor`).

In [8]:
PI = 3.141592653589793
df = (df
      .withColumn("label", F.col("Class"))
      .withColumn("hora", F.floor(F.col("Time") / 3600) % 24)
      .withColumn("hora_sin", F.sin(2 * F.lit(PI) * F.col("hora") / 24))
      .withColumn("hora_cos", F.cos(2 * F.lit(PI) * F.col("hora") / 24))
      .withColumn("periodo_dia", F.when(F.col("hora") < 6, "madrugada")
                                  .when(F.col("hora") < 12, "manha")
                                  .when(F.col("hora") < 18, "tarde")
                                  .otherwise("noite"))
      .withColumn("log_valor", F.log1p(F.col("Amount")))
      .withColumn("faixa_valor", F.when(F.col("Amount") == 0, "zero")
                                  .when(F.col("Amount") < 10, "ate_10")
                                  .when(F.col("Amount") < 100, "10_100")
                                  .when(F.col("Amount") < 500, "100_500")
                                  .otherwise("acima_500"))
      .withColumn("id", F.monotonically_increasing_id()))

cat_cols = ["periodo_dia", "faixa_valor"]
num_cols = [f"V{i}" for i in range(1, 29)] + ["log_valor", "hora_sin", "hora_cos"]

dados = df.select("id", *cat_cols, *num_cols, "label").persist(StorageLevel.MEMORY_AND_DISK)
n_dados = dados.count()          # materializa o cache (os ids ficam estáveis)
print(f"Registros prontos para modelagem: {n_dados:,}")

dados.groupBy("periodo_dia").agg(F.count("*").alias("n"), F.round(F.avg("label") * 100, 3).alias("%_fraude")).orderBy("periodo_dia").show()
dados.groupBy("faixa_valor").agg(F.count("*").alias("n"), F.round(F.avg("label") * 100, 3).alias("%_fraude")).orderBy("faixa_valor").show()

Registros prontos para modelagem: 283,726
+-----------+-----+--------+
|periodo_dia|    n|%_fraude|
+-----------+-----+--------+
|  madrugada|23842|   0.482|
|      manha|70643|   0.167|
|      noite|93120|   0.115|
|      tarde|96121|   0.138|
+-----------+-----+--------+

+-----------+------+--------+
|faixa_valor|     n|%_fraude|
+-----------+------+--------+
|    100_500| 47817|    0.19|
|     10_100|129577|   0.085|
|  acima_500|  9459|   0.359|
|     ate_10| 95065|   0.224|
|       zero|  1808|   1.383|
+-----------+------+--------+



## 5. Modelagem

### 5.1 Divisão dos dados
Como a fraude é raríssima, uma divisão aleatória simples poderia deixar pouquíssimos casos positivos em algum conjunto. Usamos **divisão estratificada** (`sampleBy`), que mantém a proporção de fraudes em todos os conjuntos:
- **Teste (20%):** só na avaliação final.
- **Treino (60%):** ajuste do Pipeline, dos pesos de classe e dos hiperparâmetros (validação cruzada).
- **Validação (20%):** só para escolher o **limiar de decisão** (ver 5.5), sem tocar no teste.

### 5.2 Desbalanceamento
Usamos **pesos de classe** (`weightCol`) calculados no treino, para que o modelo não ignore a classe rara. A métrica de seleção na validação cruzada é a **AUC-PR**, mais informativa que a AUC-ROC quando a classe positiva é rara.

In [9]:
def split_estratificado(d, frac, seed):
    a = d.sampleBy("label", {0.0: frac, 1.0: frac}, seed=seed)
    b = d.join(a.select("id"), "id", "left_anti")
    return a, b

treino_full, teste = split_estratificado(dados, 0.8, 42)
treino, validacao = split_estratificado(treino_full, 0.75, 43)
for d in (treino, validacao, teste):
    d.persist(StorageLevel.MEMORY_AND_DISK)

for nome, d in [("Treino", treino), ("Validação", validacao), ("Teste", teste)]:
    n = d.count(); p = d.filter("label = 1").count()
    print(f"{nome:10s} {n:>8,} registros | fraudes: {p:>4} ({p / n:.3%})")

# Pesos de classe a partir do TREINO apenas
n_treino = treino.count()
n_pos = treino.filter("label = 1").count()
n_neg = n_treino - n_pos
peso_pos = n_treino / (2.0 * n_pos)
peso_neg = n_treino / (2.0 * n_neg)
print(f"Pesos de classe: fraude={peso_pos:.1f}, legítima={peso_neg:.3f}")
treino_w = treino.withColumn("peso", F.when(F.col("label") == 1, peso_pos).otherwise(peso_neg))

Treino      170,864 registros | fraudes:  285 (0.167%)
Validação    56,524 registros | fraudes:   90 (0.159%)
Teste        56,338 registros | fraudes:   98 (0.174%)
Pesos de classe: fraude=299.8, legítima=0.501


In [10]:
def etapas_prep():
    """Etapas de pré-processamento (objetos novos a cada chamada)."""
    idx_cols = [c + "_idx" for c in cat_cols]
    ohe_cols = [c + "_ohe" for c in cat_cols]
    indexer = StringIndexer(inputCols=cat_cols, outputCols=idx_cols, handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols, handleInvalid="keep")
    va_num  = VectorAssembler(inputCols=num_cols, outputCol="num_vec")
    scaler  = StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=True, withStd=True)
    va_all  = VectorAssembler(inputCols=["num_scaled"] + ohe_cols, outputCol="features")
    return [indexer, encoder, va_num, scaler, va_all]

ev_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
ev_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
ev_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

def avaliar_pred(pred, nome):
    """pred: label, prediction, rawPrediction (vetor ou score numérico)."""
    pred = pred.select("label", "prediction", "rawPrediction").persist(StorageLevel.MEMORY_AND_DISK)
    cm = {(r["label"], r["prediction"]): r["count"] for r in pred.groupBy("label", "prediction").count().collect()}
    tn, fp, fn, tp = (cm.get((0.0, 0.0), 0), cm.get((0.0, 1.0), 0), cm.get((1.0, 0.0), 0), cm.get((1.0, 1.0), 0))
    prec1 = tp / (tp + fp) if tp + fp else 0.0
    rec1  = tp / (tp + fn) if tp + fn else 0.0
    res = {"modelo": nome,
           "AUC_ROC": ev_auc.evaluate(pred), "AUC_PR": ev_pr.evaluate(pred),
           "F1_ponderado": ev_f1.evaluate(pred),
           "F1_fraude": 2 * prec1 * rec1 / (prec1 + rec1) if prec1 + rec1 else 0.0,
           "precisao": prec1, "recall": rec1,
           "tn": tn, "fp": fp, "fn": fn, "tp": tp}
    pred.unpersist()
    return res

### 5.3 Referência (baseline)
Classificador trivial que **sempre prevê "legítima"**. Ele acerta mais de 99,8% (acurácia enganosa!), mas **não detecta nenhuma fraude**: AUC-ROC = 0,5, AUC-PR igual à prevalência e F1 da fraude igual a 0. É o motivo de não usarmos acurácia como métrica.

In [11]:
b1 = teste.withColumn("prediction", F.lit(0.0)).withColumn("rawPrediction", F.lit(0.0))
res_b1 = avaliar_pred(b1, "Baseline (sempre 'legítima')")
n_teste = teste.count()
prev_teste = teste.filter("label = 1").count() / n_teste
print(f"Prevalência de fraude no teste: {prev_teste:.4%}")
print(f"Baseline: acurácia = {(res_b1['tn'] / n_teste):.4%} | F1 da fraude = {res_b1['F1_fraude']:.3f} | AUC-PR = {res_b1['AUC_PR']:.4f}")

Prevalência de fraude no teste: 0.1740%
Baseline: acurácia = 99.8260% | F1 da fraude = 0.000 | AUC-PR = 0.0017


### 5.4 Ajuste de hiperparâmetros
**Validação cruzada com 3 folds** (`CrossValidator`) sobre o treino, otimizando **AUC-PR**. O Pipeline completo é reajustado em cada fold.
- **Regressão Logística:** `regParam` ∈ {0,0; 0,01; 0,1} × `elasticNetParam` ∈ {0,0; 0,5}
- **GBT:** `maxDepth` ∈ {3, 5} × `maxIter` ∈ {20, 40}

In [12]:
def ajustar_cv(clf, grid, nome):
    cv = CrossValidator(estimator=Pipeline(stages=etapas_prep() + [clf]), estimatorParamMaps=grid,
                        evaluator=ev_pr, numFolds=3, parallelism=2, seed=42)
    m = cv.fit(treino_w)
    print(f"{nome} — AUC-PR médio (CV) por combinação:", [round(x, 4) for x in m.avgMetrics])
    return m.bestModel

lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="peso", maxIter=100)
grid_lr = (ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1])
                             .addGrid(lr.elasticNetParam, [0.0, 0.5]).build())
modelo_lr = ajustar_cv(lr, grid_lr, "Regressão Logística")
m = modelo_lr.stages[-1]
print(f"Melhor LR -> regParam={m.getRegParam()}, elasticNetParam={m.getElasticNetParam()}")

gbt = GBTClassifier(featuresCol="features", labelCol="label", weightCol="peso", seed=42)
grid_gbt = (ParamGridBuilder().addGrid(gbt.maxDepth, [3, 5])
                              .addGrid(gbt.maxIter, [20, 40]).build())
modelo_gbt = ajustar_cv(gbt, grid_gbt, "GBT")
m = modelo_gbt.stages[-1]
print(f"Melhor GBT -> maxDepth={m.getMaxDepth()}, maxIter={m.getMaxIter()}")

Regressão Logística — AUC-PR médio (CV) por combinação: [np.float64(0.7057), np.float64(0.7057), np.float64(0.6491), np.float64(0.6614), np.float64(0.6728), np.float64(0.6685)]
Melhor LR -> regParam=0.0, elasticNetParam=0.5
GBT — AUC-PR médio (CV) por combinação: [np.float64(0.6873), np.float64(0.6644), np.float64(0.6456), np.float64(0.5942)]
Melhor GBT -> maxDepth=3, maxIter=20


### 5.5 Escolha do limiar de decisão
Com pesos de classe altos, o limiar padrão de 0,5 tende a gerar **muitos alertas falsos**. O limiar é uma decisão de negócio (custo de fraude não detectada × custo de revisar um alerta) e é escolhido no conjunto de **validação** para maximizar o F1 da classe fraude. O **teste não é usado** nessa escolha.

In [13]:
LIMIARES = [round(0.05 * k, 2) for k in range(1, 20)]

def probabilidades(modelo, d):
    return (modelo.transform(d)
                  .select("label", vector_to_array("probability")[1].alias("p1"))
                  .persist(StorageLevel.MEMORY_AND_DISK))

def escolher_limiar(p):
    exprs = [F.sum((F.col("label") == 1).cast("int")).alias("pos")]
    for k, t in enumerate(LIMIARES):
        pr = F.col("p1") >= t
        exprs += [F.sum((pr & (F.col("label") == 1)).cast("int")).alias(f"tp_{k}"),
                  F.sum((pr & (F.col("label") == 0)).cast("int")).alias(f"fp_{k}")]
    r = p.agg(*exprs).first()
    melhor_t, melhor_f1 = 0.5, -1.0
    for k, t in enumerate(LIMIARES):
        tp, fp = r[f"tp_{k}"], r[f"fp_{k}"]
        fn = r["pos"] - tp
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
        if f1 > melhor_f1:
            melhor_t, melhor_f1 = t, f1
    return melhor_t, melhor_f1

def avaliar_limiar(p, limiar, nome):
    pred = (p.withColumn("prediction", F.when(F.col("p1") >= limiar, 1.0).otherwise(0.0))
              .withColumnRenamed("p1", "rawPrediction"))
    return avaliar_pred(pred, nome)

p_val_lr,  p_val_gbt  = probabilidades(modelo_lr, validacao), probabilidades(modelo_gbt, validacao)
t_lr,  f1_val_lr  = escolher_limiar(p_val_lr)
t_gbt, f1_val_gbt = escolher_limiar(p_val_gbt)
print(f"Limiar escolhido (validação): LR = {t_lr} (F1 val = {f1_val_lr:.3f}) | GBT = {t_gbt} (F1 val = {f1_val_gbt:.3f})")

Limiar escolhido (validação): LR = 0.95 (F1 val = 0.491) | GBT = 0.95 (F1 val = 0.731)


## 6. Avaliação e interpretação
Métricas para classificação: **AUC**, **F1** e **matriz de confusão**, com atenção ao desbalanceamento. Reportamos também **AUC-PR**, **precisão** e **recall** da classe fraude (a acurácia é omitida por ser enganosa aqui). Tudo no **conjunto de teste**.

In [14]:
p_teste_lr, p_teste_gbt = probabilidades(modelo_lr, teste), probabilidades(modelo_gbt, teste)

resultados = [
    res_b1,
    avaliar_limiar(p_teste_lr,  0.5,   "Reg. Logística (limiar 0,5)"),
    avaliar_limiar(p_teste_lr,  t_lr,  f"Reg. Logística (limiar {t_lr})"),
    avaliar_limiar(p_teste_gbt, 0.5,   "GBT (limiar 0,5)"),
    avaliar_limiar(p_teste_gbt, t_gbt, f"GBT (limiar {t_gbt})"),
]
res_lr_t, res_gbt_t = resultados[2], resultados[4]

print(f"{'Modelo':32s} {'AUC':>7s} {'AUC-PR':>8s} {'F1 fraude':>10s} {'Precisão':>9s} {'Recall':>8s} {'F1 pond.':>9s}")
for r in resultados:
    print(f"{r['modelo']:32s} {r['AUC_ROC']:7.4f} {r['AUC_PR']:8.4f} {r['F1_fraude']:10.4f} {r['precisao']:9.4f} {r['recall']:8.4f} {r['F1_ponderado']:9.4f}")

Modelo                               AUC   AUC-PR  F1 fraude  Precisão   Recall  F1 pond.
Baseline (sempre 'legítima')      0.5000   0.0017     0.0000    0.0000   0.0000    0.9974
Reg. Logística (limiar 0,5)       0.9741   0.6897     0.1139    0.0609   0.8673    0.9866
Reg. Logística (limiar 0.95)      0.9741   0.6897     0.5280    0.3795   0.8673    0.9978
GBT (limiar 0,5)                  0.9636   0.7226     0.0988    0.0523   0.8878    0.9841
GBT (limiar 0.95)                 0.9636   0.7226     0.7667    0.8415   0.7041    0.9992


In [ ]:
# Matrizes de confusão (linhas = real, colunas = previsto)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, r in zip(axes, [res_b1, res_lr_t, res_gbt_t]):
    m = [[r["tn"], r["fp"]], [r["fn"], r["tp"]]]
    ax.imshow(m, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{m[i][j]:,}", ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["legítima", "fraude"]); ax.set_yticklabels(["legítima", "fraude"])
    ax.set_xlabel("Previsto"); ax.set_ylabel("Real"); ax.set_title(r["modelo"], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Importância dos atributos (GBT)
def nomes_features(modelo, d):
    meta = modelo.transform(d.limit(20)).schema["features"].metadata["ml_attr"]["attrs"]
    nomes = {}
    for grupo in meta.values():
        for a in grupo:
            nome = a["name"]
            if nome.startswith("num_scaled_"):
                nome = num_cols[int(nome.split("_")[-1])]
            nomes[a["idx"]] = nome
    return nomes

try:
    nomes = nomes_features(modelo_gbt, teste)
    imp = modelo_gbt.stages[-1].featureImportances
    top = sorted(((nomes.get(int(i), f"f{i}"), float(v)) for i, v in zip(imp.indices, imp.values)),
                 key=lambda x: -x[1])[:12]
    plt.figure(figsize=(7, 4))
    plt.barh([t[0] for t in top][::-1], [t[1] for t in top][::-1])
    plt.title("Top 12 atributos — GBT"); plt.xlabel("Importância"); plt.tight_layout(); plt.show()
except Exception as e:
    print("Não foi possível extrair os nomes das features:", e)

### 6.1 Interpretação (gerada com os números desta execução)

In [17]:
n_fraudes_teste = res_b1["fn"]
melhor = max([res_lr_t, res_gbt_t], key=lambda r: r["AUC_PR"])
print(f"""
RESUMO DOS RESULTADOS (teste: {n_teste:,} transações, {n_fraudes_teste} fraudes = {prev_teste:.3%})
- Baseline ('sempre legítima'): acerta {res_b1['tn'] / n_teste:.2%} das transações, mas detecta 0 fraudes (F1 = 0, AUC-PR = {res_b1['AUC_PR']:.4f}).
- Reg. Logística: AUC-PR = {res_lr_t['AUC_PR']:.3f}, F1 da fraude = {res_lr_t['F1_fraude']:.3f} (precisão {res_lr_t['precisao']:.3f}, recall {res_lr_t['recall']:.3f}).
- GBT:            AUC-PR = {res_gbt_t['AUC_PR']:.3f}, F1 da fraude = {res_gbt_t['F1_fraude']:.3f} (precisão {res_gbt_t['precisao']:.3f}, recall {res_gbt_t['recall']:.3f}).
- Melhor por AUC-PR: {melhor['modelo']}.
- Em termos práticos, com o melhor modelo: das {n_fraudes_teste} fraudes do teste, {melhor['tp']} foram detectadas e {melhor['fn']} escaparam.
  Foram gerados {melhor['fp']:,} alertas falsos (de {n_teste - n_fraudes_teste:,} transações legítimas),
  ou seja, a cada 100 alertas, cerca de {100 * melhor['precisao']:.0f} são fraudes reais.
""")


RESUMO DOS RESULTADOS (teste: 56,338 transações, 98 fraudes = 0.174%)
- Baseline ('sempre legítima'): acerta 99.83% das transações, mas detecta 0 fraudes (F1 = 0, AUC-PR = 0.0017).
- Reg. Logística: AUC-PR = 0.690, F1 da fraude = 0.528 (precisão 0.379, recall 0.867).
- GBT:            AUC-PR = 0.723, F1 da fraude = 0.767 (precisão 0.841, recall 0.704).
- Melhor por AUC-PR: GBT (limiar 0.95).
- Em termos práticos, com o melhor modelo: das 98 fraudes do teste, 69 foram detectadas e 29 escaparam.
  Foram gerados 13 alertas falsos (de 56,240 transações legítimas),
  ou seja, a cada 100 alertas, cerca de 84 são fraudes reais.



### 6.2 O que os resultados significam para a pergunta de negócio
- O baseline mostra por que a **acurácia não serve**: prever tudo "legítima" dá ~99,8% e é inútil. Por isso a comparação relevante é feita por AUC-PR, F1 da fraude, precisão e recall.
- A comparação entre o limiar 0,5 e o limiar escolhido na validação mostra o **trade-off** central do problema: aumentar o limiar reduz alertas falsos (maior precisão), mas deixa passar mais fraudes (menor recall). O ponto certo depende do custo relativo dos dois erros, uma decisão de negócio. Um banco com revisão manual barata aceita muitos alertas; um com bloqueio automático e clientes sensíveis prefere precisão alta.
- A importância dos atributos mostra quais componentes (`V*`) mais separam fraudes de transações legítimas. Como eles são anonimizados (PCA), **não é possível explicar** o comportamento em termos de negócio, o que é uma limitação para auditoria e regulação.

### 6.3 Limitações
1. **Poucos casos positivos:** o teste tem apenas ~100 fraudes, então as métricas têm **alta variância** (poucas fraudes a mais ou a menos mudam bastante o recall). Seriam necessários mais dados ou intervalos de confiança (por exemplo, com *bootstrap*) para comparar modelos com segurança.
2. **Atributos anonimizados (PCA):** `V1`–`V28` não são interpretáveis, e não há atributos de histórico do cliente/cartão (frequência recente, país, estabelecimento), que costumam ser os mais úteis em produção.
3. **Período curto e antigo:** dois dias de 2013 de cartões europeus. Padrões de fraude mudam (*concept drift*) e podem não valer hoje, nem em outros países.
4. **Divisão estratificada aleatória:** em produção o modelo prevê o futuro; o correto seria uma divisão **temporal**. Aqui isso seria inviável, pois há poucos dias e poucas fraudes.
5. **Custos não modelados:** o limiar maximiza o F1, que trata erros igualmente; o ideal seria otimizar o **custo esperado** (valor da fraude × custo de revisão).
6. **Escala:** a base é uma amostra pequena; o comportamento dos modelos e o tempo de treino em volumes reais (centenas de milhões de transações) não foram avaliados.

## 7. Persistência
O Pipeline completo (indexadores, encoders, assembler, scaler e classificador) é salvo com `save()` do Spark ML, assim o modelo carregado reaplica o mesmo pré-processamento. Salvamos o melhor modelo por AUC-PR junto com o limiar escolhido.

In [18]:
if melhor is res_gbt_t:
    modelo_final, limiar_final, nome_final = modelo_gbt, t_gbt, "GBT"
else:
    modelo_final, limiar_final, nome_final = modelo_lr, t_lr, "Regressão Logística"

CAMINHO_MODELO = os.path.abspath("modelos/modelo_fraude_cartao")
modelo_final.write().overwrite().save(CAMINHO_MODELO)
with open(f"{os.path.dirname(CAMINHO_MODELO)}/limiar.json", "w") as f:
    json.dump({"modelo": nome_final, "limiar": limiar_final}, f)
print("Modelo salvo em:", CAMINHO_MODELO, "| tipo:", nome_final, "| limiar:", limiar_final)

# Verificação: recarregar e conferir que as previsões são idênticas
recarregado = PipelineModel.load(CAMINHO_MODELO)
amostra = teste.limit(1000)
a = [r.prediction for r in modelo_final.transform(amostra).select("prediction").collect()]
b = [r.prediction for r in recarregado.transform(amostra).select("prediction").collect()]
print("Previsões idênticas após recarregar:", a == b)

Modelo salvo em: /content/modelos/modelo_fraude_cartao | tipo: GBT | limiar: 0.95
Previsões idênticas após recarregar: True


## 8. Conclusão
- Construímos, com PySpark, um pipeline completo de classificação para **detecção de fraude em cartão**: ingestão com schema manual, limpeza (inclusive duplicatas) e exploração em Spark, engenharia de atributos (codificação cíclica da hora, `log` do valor, faixas de valor) em `Pipeline`, **validação cruzada** para hiperparâmetros e avaliação em teste com AUC-ROC, AUC-PR, F1 e matriz de confusão.
- O forte desbalanceamento (0,17%) foi tratado com divisão **estratificada**, **pesos de classe**, seleção por **AUC-PR** e **ajuste de limiar** em um conjunto de validação separado, sem tocar no teste. Isso evita vazamento de dados.
- O baseline mostra que a acurácia é enganosa, e os modelos o superam nas métricas adequadas (ver tabela da seção 6).
- O problema se justifica como "dados massivos" pelo **cenário** (bilhões de transações, classe rara, decisão em tempo real, concept drift), embora a base usada seja uma amostra.

**Possíveis melhorias**
- Atributos de histórico por cartão com *window functions* (transações nas últimas horas, valor médio, mudança de país) em base maior.
- Divisão temporal e monitoramento de *drift*; retreino periódico.
- Otimização do limiar por **custo esperado** e intervalos de confiança por *bootstrap*.
- Outros modelos (Random Forest, XGBoost em Spark) e técnicas de reamostragem (subamostragem da classe majoritária).
- Execução em cluster e *scoring* em *streaming* (Structured Streaming).

In [19]:
spark.stop()